# COMP8240 — Reproducing BERTopic: constructed news dataset (v2) pipeline

Runs every step the final report needs for the **newly constructed dataset**, in order,
on your own collected articles, and writes all numbers, tables and figures to
`results/v2/report/` (see `REPORT_NUMBERS.md` there).

**Before you start**
1. *Runtime → Change runtime type → T4 GPU.*
2. Put your project folder on Google Drive (e.g. `MyDrive/COMP8240_project`) containing:
   - `src/` — **the updated scripts from this package** (copy them over your old ones),
   - `ANNOTATION_GUIDE.md`,
   - `data/v2/articles_raw.csv` and `data/v2/collection_log.csv` (from your daily collection),
   - optionally `results/constructed_dataset_assignments.csv` (v1 result, shown as a reference row).
3. Run the cells top to bottom. The notebook **stops at Step 3** so you can label events;
   come back and continue from Step 4 once the sheets are labelled.

Steps: 1 setup · 2 preprocess · 3 annotation sheets (manual labelling) · 4 agreement ·
5 BERTopic sweep · 6 LDA baseline · 7 events-vs-outlets evaluation · 8 LLM-as-judge ·
9 report tables · 10 (optional) refresh replication numbers · 11 download results

## 1 · Setup

In [ ]:
# ---- settings -------------------------------------------------------------
PROJECT_DIR = "/content/drive/MyDrive/COMP8240_project"   # folder with src/ and data/v2/
START, END = None, None        # optional date window for the dataset, e.g. "2026-10-10", "2026-11-05"
SEEDS = [1, 2, 3]              # the paper averages 3 runs
MAIN_MTS = 5                   # main min_topic_size; 3 and 10 are run once as a sensitivity check
RUN_MPNET = True               # also run the paper's embedding model (all-mpnet-base-v2) on capped text
JUDGE_PROVIDER = "hf"          # "hf" = free open model on the Colab GPU; "anthropic" needs ANTHROPIC_API_KEY
OFFLINE_TEST = False           # True = fast pipeline test with a TF-IDF stand-in + mock judge (NOT reportable)
# ----------------------------------------------------------------------------
import os, subprocess, sys
try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
os.chdir(PROJECT_DIR)
print("Working in", os.getcwd(), "| contents:", sorted(os.listdir(".")))
assert os.path.isfile("src/v2_common.py"), "src/ not found -- check PROJECT_DIR"
assert os.path.isfile("data/v2/articles_raw.csv"), "data/v2/articles_raw.csv not found"

In [ ]:
if IN_COLAB:
    !pip -q install bertopic sentence-transformers feedparser accelerate tabulate

In [ ]:
def run(cmd):
    """Run a project script, show its output, stop the notebook if it fails."""
    print(f"$ {cmd}")
    p = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    out = "\n".join(l for l in p.stdout.splitlines() if "it/s]" not in l)
    print(out[-6000:])
    if p.returncode != 0:
        print(p.stderr[-4000:])
        raise RuntimeError(f"failed: {cmd}")

OFF = " --offline" if OFFLINE_TEST else ""
try:
    import torch; print("GPU available:", torch.cuda.is_available())
except ImportError:
    print("torch not installed (fine for OFFLINE_TEST)")

## 2 · Preprocess the collected articles (`08_preprocess_articles.py`)
Cleans HTML/boilerplate, removes duplicates and non-story items (live blogs, videos, quizzes),
and builds two text versions: **full** (title + summary) and **capped** (title + first sentence,
max 40 words) to control the outlet length confound found in v1.

In [ ]:
win = (f" --start {START}" if START else "") + (f" --end {END}" if END else "")
run(f"python src/08_preprocess_articles.py{win}")

## 3 · Annotation sheets (`09_annotation_sheet.py`) — manual step

1. The next cell creates `data/v2/annotations_A.csv` (≈250 articles from whole days) and
   `data/v2/annotations_B.csv` (75 of them, shuffled, no hints) for a **second annotator**
   (a classmate, friend or family member — the brief asks for a measure of agreement).
2. Open each in Google Sheets/Excel, fill `event_id` (+ `event_description`) for every row
   following `ANNOTATION_GUIDE.md`, save as CSV under the same name in `data/v2/`.
3. Re-run the status cell. Continue to Step 4 when both sheets are fully labelled.

Existing sheets are never overwritten (labels are safe).

In [ ]:
run("python src/09_annotation_sheet.py --generate --n 250")
run("python src/09_annotation_sheet.py --second --n_b 75")

In [ ]:
run("python src/09_annotation_sheet.py --status")

## 4 · Inter-annotator agreement (`10_annotation_agreement.py`)
Cohen's kappa on pairwise same-event decisions, plus ARI.

In [ ]:
run("python src/10_annotation_agreement.py")

## 5 · BERTopic sweep (`11_bertopic_articles.py`)
Main setting: `min_topic_size=5`, 3 seeds, full and capped text (→ mean ± sd like the paper).
Sensitivity: `min_topic_size` 3 and 10 (seed 1). Optionally the paper's `all-mpnet-base-v2` embeddings.

In [ ]:
for text in ["full", "capped"]:
    for s in SEEDS:
        run(f"python src/11_bertopic_articles.py --text {text} --min_topic_size {MAIN_MTS} --seed {s}{OFF}")
for mts in [3, 10]:
    run(f"python src/11_bertopic_articles.py --text capped --min_topic_size {mts} --seed {SEEDS[0]}{OFF}")
if RUN_MPNET and not OFFLINE_TEST:
    for s in SEEDS:
        run(f"python src/11_bertopic_articles.py --text capped --min_topic_size {MAIN_MTS} --seed {s} --embedding all-mpnet-base-v2")

## 6 · LDA baseline (`12_lda_baseline.py`)
Same articles, `k` = mean number of topics BERTopic found in the main setting, so the two are comparable.

In [ ]:
import pandas as pd
rs = pd.read_csv("results/v2/runs_summary.csv")
for text in ["full", "capped"]:
    main = rs[(rs.model == "BERTopic") & (rs.text == text) & (rs.min_topic_size == MAIN_MTS)
              & rs.run.str.contains("seed") & ~rs.run.str.contains("mpnet")]
    k = int(round(main.n_topics.mean()))
    for s in SEEDS:
        run(f"python src/12_lda_baseline.py --text {text} --k {k} --seed {s}")

## 7 · Events or outlets? (`13_evaluate_event_vs_outlet.py`)
Outlet effect on all articles (NMI + permutation test) and event tracking on the labelled
articles (NMI/ARI vs your event IDs, and the pair rates SE/XO, DE/SO, DE/XO).

In [ ]:
run("python src/13_evaluate_event_vs_outlet.py")
from IPython.display import Image, display
if os.path.isfile("results/v2/event_vs_outlet_pairs.png"):
    display(Image("results/v2/event_vs_outlet_pairs.png"))

## 8 · LLM-as-judge (`15_llm_judge.py`)
Q1 rates each topic's words 1–3 (Stammbach et al., 2023). Q2 asks whether a topic is one **EVENT**,
a broad **THEME** or **MIXED** — checked against your human labels via topic purity.
The default judge is `Qwen/Qwen2.5-3B-Instruct` on the Colab GPU (free, no key).
Judged: main-setting runs (seed 1) for both models and texts.

In [ ]:
prov = "mock" if OFFLINE_TEST else JUDGE_PROVIDER
rs = pd.read_csv("results/v2/runs_summary.csv")
pick = rs[rs.run.str.contains(f"seed{SEEDS[0]}") &
          ((rs.model == "LDA") | (pd.to_numeric(rs.min_topic_size, errors="coerce") == MAIN_MTS))].run.tolist()
run(f"python src/15_llm_judge.py --provider {prov} --runs " + " ".join(pick))

## 9 · Report tables and figures (`14_report_tables.py`)
Everything for the report in `results/v2/report/` — CSV, LaTeX tables for Overleaf, figures and `REPORT_NUMBERS.md`.

In [ ]:
run("python src/14_report_tables.py")
display(Image("results/v2/report/articles_per_day.png"))

## 10 · (Optional) Refresh the replication numbers
The scripts now apply `min_df` per document instead of per topic (see `utils.py` →
`document_level_vectorizer`). Results produced before this fix used fewer topic-unique words,
so re-run the replication with 3 seeds so the report's replication table matches the code in your repo.
This takes roughly 1–2 hours on a T4. Skip it if time is short and say so in the report.

In [ ]:
REFRESH_REPLICATION = False
if REFRESH_REPLICATION:
    for k in [10, 50]:
        for s in SEEDS:
            run(f"python src/01_reproduce_original.py --nr_topics {k} --seed {s}{OFF}")
            run(f"python src/06_trumps_tweets.py --nr_topics {k} --seed {s}{OFF}")
    for s in SEEDS:
        run(f"python src/05_bbc_news.py --nr_topics 5 --seed {s}{OFF}")
        run(f"python src/02_new_existing_dataset.py --seed {s}{OFF}")
    for f in ["20newsgroups", "bbcnews", "trumptweets"]:
        p = f"results/{f}_sweep_summary.csv"
        if os.path.isfile(p):
            d = pd.read_csv(p)
            print(f, "\n", d.groupby("nr_topics_requested")[["npmi_coherence", "topic_diversity", "nr_topics_found"]]
                  .agg(["mean", "std"]).round(3).to_string())

### 10b · LLM-as-judge on the replication and Amazon topics
Same 1–3 interpretability question as Step 8, so the judge's view of the new data can be compared with the original datasets (the brief asks how new results relate to the replication). Uses the topic files written by Step 10 (the last seed run for each setting).

In [ ]:
prov = "mock" if OFFLINE_TEST else JUDGE_PROVIDER
import glob
extra = [f for f in glob.glob("results/*_topics_*.csv")]
print(len(extra), "topic files:", extra)
if extra and REFRESH_REPLICATION:   # only judge topics made with the fixed code
    run(f"python src/15_llm_judge.py --provider {prov} --extra_only --extra \"results/*_topics_*.csv\"")
    print(pd.read_csv("results/v2/llm_judge/llm_judge_summary_extra.csv").to_string())

## 11 · Download all results

In [ ]:
import shutil
shutil.make_archive("/content/results_v2", "zip", "results")
if IN_COLAB:
    from google.colab import files
    files.download("/content/results_v2.zip")
print(open("results/v2/report/REPORT_NUMBERS.md").read()[:3000])